# Theory: Gaussian Beam Optics

**Author:** Tyler Celotta | UAF

## Abstract
The purpose of this notebook is to work through the theory and experimental design for optimizing second-harmonic generation (SHG) at 1064 nm using a periodically poled lithium niobate (PPLN) crystal (50 × 2 × 0.5 mm³). We define Gaussian beam parameters—including beam waist, Rayleigh range, divergence, and confocal parameter—to determine the optimal focusing geometry. 

The Boyd-Kleinman condition for maximum CW SHG efficiency ($L/b = 2.84$) sets a target focused waist of approximately 54.6 µm, but experimental constraints redefine a target of 77–80 µm ($L/b \approx 1.3-1.4$). 

Starting from a collimated 0.56 mm beam at 1064 nm with a measured divergence of 1.21 mRad, the collimator waist is estimated to be $\omega_{01} \approx 280$ µm, located 6.2 mm behind the collimator face. Because a waist radius of roughly 0.279 mm would result in a beam larger than the crystal itself can accommodate, we use an ABCD ray-transfer matrix to propagate the beam, constructing equations relating a chosen lens focal length $f$ to the required lens position $d_{1}$ and image distance $d_{2}$.


## 1. Experiment Specifics
* **Wavelength:** 1064 nm
* **Collimator output beam size:** 0.56 mm (1/e² diameter)
* **PPLN crystal:** 50 × 2 × 0.5 mm³
* **Previous best measured output:** 57 mW green, PPLN at 59.5°C
* **Previous calculated waist:** ~45 µm
* **Desired next target:** ~77–80 µm

In [1]:
import numpy as np
import math

In [2]:
# Global Parameters
wavelength = 1064e-9  # m
collimator_diam = 0.56e-3  # m
crystal_length = 50e-3  # m
measured_divergence = 1.21e-3  # rad

## 2. Definitions and Equations

**Beam Waist ($\omega_0$):** The portion of the beam where the radius is minimized. The tighter the focus, the smaller the waist, but this introduces larger divergence:
$$\omega^2(z)=\omega_0^2\left[1+\left(\frac{\lambda z}{\pi \omega_0^2}\right)^2\right]$$

**Rayleigh Range ($z_0$):** The distance at which the cross-sectional area of the beam is no more than twice that of the beam waist:
$$z_0=\frac{\pi \omega_0^2}{\lambda}$$

**Beam Divergence ($\theta$):** The far-field expansion half-angle:
$$\theta=\frac{\lambda}{\pi \omega_0}$$

**Confocal Parameter ($b$):** The total length of the focused region around the beam waist ($b = 2z_0$).

## 3. Initial Beam Parameters & Collimator Estimation

Repeating the analysis without assuming the beam waist is exactly at the collimator face (using the measured 1.21 mRad divergence).

1. Recover the true waist from the measured divergence.
2. Calculate the distance $z$ from the true waist to the collimator face where the beam radius is 0.28 mm.
3. Recalculate the true Rayleigh range.

In [3]:
# 1. True initial waist from measured divergence
omega_01 = wavelength / (np.pi * measured_divergence)
print(f"True Collimator Waist (omega_01): {omega_01 * 1e6:.1f} µm")

# 2. Distance from waist to collimator face
omega_aperture = collimator_diam / 2
# Solving the omega(z) equation for z
z_collimator = np.sqrt((omega_aperture**2 / omega_01**2) - 1) * (np.pi * omega_01**2) / wavelength
print(f"Distance from waist to face: {z_collimator * 1e3:.2f} mm")

# 3. True Rayleigh range of the initial beam
z_01 = (np.pi * omega_01**2) / wavelength
print(f"Initial Rayleigh Range (z_01): {z_01 * 100:.2f} cm")

True Collimator Waist (omega_01): 279.9 µm
Distance from waist to face: 6.11 mm
Initial Rayleigh Range (z_01): 23.13 cm


## 4. Boyd-Kleinman Focusing Targets

To recover a desired focused waist ($\omega_{02}$), we solve for the ratio $\xi \equiv L/b$. The theoretical optimum for maximum CW SHG efficiency is $L/b = 2.84$.

$$b = 2z_{02} = \frac{2\pi \omega_{02}^2}{\lambda}$$
$$\xi = \frac{L \lambda}{2\pi \omega_{02}^2}$$

In [4]:
targets_um = [45, 54.6, 77, 80] # µm

print("Boyd-Kleinman Ratios (L/b) for target waists:")
print("-" * 50)
for target in targets_um:
    omega_02 = target * 1e-6
    b = (2 * np.pi * omega_02**2) / wavelength
    xi = crystal_length / b
    print(f"Target Waist: {target:>4.1f} µm | L/b = {xi:.3f}")

Boyd-Kleinman Ratios (L/b) for target waists:
--------------------------------------------------
Target Waist: 45.0 µm | L/b = 4.181
Target Waist: 54.6 µm | L/b = 2.840
Target Waist: 77.0 µm | L/b = 1.428
Target Waist: 80.0 µm | L/b = 1.323


## 5. Lens Selection and Placement

Because the beam waist ($\omega_{01} \approx 280$ µm) is the *minimum* radius of the laser, it will scrape the inside of the 2 mm PPLN crystal if uncorrected. We must use a lens. 

Using ABCD matrix derivations for free-space and a thin lens, the equations for lens placement ($d_1$) and image distance ($d_2$) are:

**Magnification ($\alpha$):**
$$\alpha=\frac{\omega_{02}}{\omega_{01}}$$

**Distance to Lens ($d_1$):**
$$|d_1|=\sqrt{\left(\frac{f}{\alpha}\right)^2-z_{01}^2}+f$$

**Distance to Focus ($d_2$):**
$$d_2 = f + \alpha^2(d_1 - f)$$

In [6]:
# Set your target waist and chosen lab focal length here
target_omega_02 = 80e-6  # 80 µm
focal_length = float(input("Enter the lens focal length (in mm): ")) * 1e-3 

# Calculate Magnification
alpha = target_omega_02 / omega_01

# Calculate d1 (Distance from initial waist to lens)
inside_sqrt = (focal_length / alpha)**2 - z_01**2

if inside_sqrt >= 0:
    d1 = np.sqrt(inside_sqrt) + focal_length
    # Calculate d2 (Distance from lens to new focus inside crystal)
    d2 = focal_length + (alpha**2) * (d1 - focal_length)
    
    # Calculate physical distance from the collimator face to the lens
    # Note: z_collimator was calculated in Cell 6 as a positive distance
    d1_face = d1 - z_collimator 
    
    print(f"Magnification (alpha): {alpha:.4f}")
    print(f"Lens focal length: {focal_length * 1000:.1f} mm")
    print("-" * 60)
    print(f"Optical Distance (Waist to Lens, d1): {d1 * 1000:.2f} mm")
    print(f"--> Physical Distance (Collimator FACE to Lens): {d1_face * 1000:.2f} mm")
    print("-" * 60)
    print(f"Distance from Lens to Focus (d2): {d2 * 1000:.2f} mm")
else:
    print(f"No valid real solution for a {focal_length * 1000:.1f} mm lens.")
    print("Try a lens with a longer focal length.")

Magnification (alpha): 0.2858
Lens focal length: 88.3 mm
------------------------------------------------------------
Optical Distance (Waist to Lens, d1): 293.08 mm
--> Physical Distance (Collimator FACE to Lens): 286.96 mm
------------------------------------------------------------
Distance from Lens to Focus (d2): 105.03 mm


## 6. Lens Selection Lab Trails

#### Trail 1:
Chosen Focal Length: $88.3$ mm

Calculated $\alpha$: $0.2858$

Calculated d1: $293.08$ mm

Physical Distance *collimator face to Lens*: $286.96$ mm

Calculated d2: $105.03$ mm

**Best Power:**


#### Trail 2:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**


#### Trail 3:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**


#### Trail 4:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**


#### Trail 5:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**

## A1. 25mm Crystal Calculations


In [9]:
# Global Parameters
wavelength = 1064e-9  # m
collimator_diam = 0.56e-3  # m
crystal_length = 25e-3  # m
measured_divergence = 1.21e-3  # rad

In [10]:
# 1. True initial waist from measured divergence
omega_01 = wavelength / (np.pi * measured_divergence)
print(f"True Collimator Waist (omega_01): {omega_01 * 1e6:.1f} µm")

# 2. Distance from waist to collimator face
omega_aperture = collimator_diam / 2
# Solving the omega(z) equation for z
z_collimator = np.sqrt((omega_aperture**2 / omega_01**2) - 1) * (np.pi * omega_01**2) / wavelength
print(f"Distance from waist to face: {z_collimator * 1e3:.2f} mm")

# 3. True Rayleigh range of the initial beam
z_01 = (np.pi * omega_01**2) / wavelength
print(f"Initial Rayleigh Range (z_01): {z_01 * 100:.2f} cm")

True Collimator Waist (omega_01): 279.9 µm
Distance from waist to face: 6.11 mm
Initial Rayleigh Range (z_01): 23.13 cm


In [11]:
targets_um = [38.815, 54.6, 65.08] # µm

print("Boyd-Kleinman Ratios (L/b) for target waists:")
print("-" * 50)
for target in targets_um:
    omega_02 = target * 1e-6
    b = (2 * np.pi * omega_02**2) / wavelength
    xi = crystal_length / b
    print(f"Target Waist: {target:>4.1f} µm | L/b = {xi:.3f}")

Boyd-Kleinman Ratios (L/b) for target waists:
--------------------------------------------------
Target Waist: 38.8 µm | L/b = 2.810
Target Waist: 54.6 µm | L/b = 1.420
Target Waist: 65.1 µm | L/b = 1.000


In [12]:
target_omega_02 = 38.815e-6  # 38.815 µm
focal_length = float(input("Enter the lens focal length (in mm): ")) * 1e-3 

# Calculate Magnification
alpha = target_omega_02 / omega_01

# Calculate d1 (Distance from initial waist to lens)
inside_sqrt = (focal_length / alpha)**2 - z_01**2

if inside_sqrt >= 0:
    d1 = np.sqrt(inside_sqrt) + focal_length
    # Calculate d2 (Distance from lens to new focus inside crystal)
    d2 = focal_length + (alpha**2) * (d1 - focal_length)
    
    # Calculate physical distance from the collimator face to the lens
    # Note: z_collimator was calculated in Cell 6 as a positive distance
    d1_face = d1 - z_collimator 
    
    print(f"Magnification (alpha): {alpha:.4f}")
    print(f"Lens focal length: {focal_length * 1000:.1f} mm")
    print("-" * 60)
    print(f"Optical Distance (Waist to Lens, d1): {d1 * 1000:.2f} mm")
    print(f"--> Physical Distance (Collimator FACE to Lens): {d1_face * 1000:.2f} mm")
    print("-" * 60)
    print(f"Distance from Lens to Focus (d2): {d2 * 1000:.2f} mm")
else:
    print(f"No valid real solution for a {focal_length * 1000:.1f} mm lens.")
    print("Try a lens with a longer focal length.")
    

Magnification (alpha): 0.1387
Lens focal length: 50.0 mm
------------------------------------------------------------
Optical Distance (Waist to Lens, d1): 326.57 mm
--> Physical Distance (Collimator FACE to Lens): 320.46 mm
------------------------------------------------------------
Distance from Lens to Focus (d2): 55.32 mm


## A2. Lens Selection Lab Trails

#### Trail 1:
Chosen Focal Length: $88.3$ mm

Calculated $\alpha$: $0.1387$

Calculated d1: $681.54$ mm

Physical Distance (*collimator face to Lens*): $675.43$ mm

Calculated d2: $99.71$ mm

**Best Power:**


#### Trail 2:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**


#### Trail 3:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**


#### Trail 4:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**


#### Trail 5:
Chosen Focal Length: $00.0$ mm

Calculated $\alpha$:

Calculated d1:

Calculated d2:

**Best Power:**

## A3 Path Length Optimization

In [13]:
def optimize_beam_path(omega_01, z_01, z_collimator, target_omega_02=65.1e-6):
    """
    Optimizes the optical path length (d1 + d2) for a given set of beam parameters.
    """
    
    # Pre-determined dictionary of standard off-the-shelf lenses (Focal lengths in mm)
    lens_dict = {
        "Lens 0.5Dia (25mm)": 25,
        "Lens 0.5Dia (30mm)": 30,
        "Lens 0.5Dia (35mm)": 35,
        "Lens 0.5Dia (40mm)": 40,
        "Lens 0.5Dia (50mm)": 50,
        "Lens 0.5Dia (60mm)": 60,
        "Lens 0.5Dia (75mm)": 75,
        "Lens 0.5Dia (100mm)": 100,
        "Lens 0.5Dia (125mm)": 125,
        "Lens 0.5Dia (150mm)": 150,
        "Lens 0.5Dia (175mm)": 175,
        "Lens 0.5Dia (200mm)": 200,
        "Lens 0.5Dia (400mm)": 400,
        "Lens 0.5Dia (500mm)": 500
    }
    
    # Calculate Magnification
    alpha = target_omega_02 / omega_01
    
    print(f"Target Waist: {target_omega_02 * 1e6:.1f} µm")
    print(f"Initial Waist: {omega_01 * 1e6:.1f} µm")
    print(f"Magnification (alpha): {alpha:.4f}\n")
    print("-" * 80)
    print(f"{'Lens Name':<15} | {'f (mm)':<8} | {'d1 (mm)':<10} | {'d2 (mm)':<10} | {'Total Path (mm)':<15}")
    print("-" * 80)

    best_setup = None
    min_total_distance = float('inf')

    for name, f_mm in lens_dict.items():
        focal_length = f_mm * 1e-3  # Convert to meters
        
        # Calculate the inner square root term
        inside_sqrt = (focal_length / alpha)**2 - z_01**2
        
        if inside_sqrt >= 0:
            sqrt_term = np.sqrt(inside_sqrt)
            
            # Gaussian beam imaging yields TWO valid d1 positions (f + sqrt and f - sqrt)
            d1_options = [focal_length + sqrt_term, focal_length - sqrt_term]
            
            for d1 in d1_options:
                # Check if the lens is physically outside the collimator
                d1_face = d1 - z_collimator
                if d1_face <= 0:
                    continue # Skip if the lens would have to be inside the collimator
                
                # Calculate d2
                d2 = focal_length + (alpha**2) * (d1 - focal_length)
                
                # Calculate total path length
                total_distance = d1 + d2
                
                print(f"{name:<15} | {f_mm:<8.1f} | {d1*1000:<10.2f} | {d2*1000:<10.2f} | {total_distance*1000:<15.2f}")
                
                # Check if this is the new optimal path
                if total_distance < min_total_distance:
                    min_total_distance = total_distance
                    best_setup = {
                        "lens": name,
                        "focal_length": f_mm,
                        "d1": d1,
                        "d1_face": d1_face,
                        "d2": d2,
                        "total": total_distance
                    }

    print("-" * 80)
    
    # Output the ultimate winner
    if best_setup:
        print(f"Best Lens:           {best_setup['lens']} (f = {best_setup['focal_length']} mm)")
        print(f"Waist to Lens (d1):  {best_setup['d1'] * 1000:.2f} mm")
        print(f"Collimator to Lens:  {best_setup['d1_face'] * 1000:.2f} mm")
        print(f"Lens to Focus (d2):  {best_setup['d2'] * 1000:.2f} mm")
        print(f"Smallest Path (d1+d2): {best_setup['total'] * 1000:.2f} mm")
    else:
        print("\nNo valid lenses found in the dictionary for these parameters.")
        print("Try lenses with longer focal lengths, or check collimator distance.")

In [14]:
import numpy as np

# Use the actual variables calculated in your earlier notebook cells
# (Ensure cells calculating omega_01, z_01, and z_collimator have been run first)

# You can change this target waist based on your Boyd-Kleinman optimums 
# For the 25mm crystal: 38.815e-6 (L/b=2.81) or 65.08e-6 (L/b=1)
desired_target_waist = 38.815e-6 

optimize_beam_path(
    omega_01=omega_01, 
    z_01=z_01, 
    z_collimator=z_collimator,
    target_omega_02=desired_target_waist
)

Target Waist: 38.8 µm
Initial Waist: 279.9 µm
Magnification (alpha): 0.1387

--------------------------------------------------------------------------------
Lens Name       | f (mm)   | d1 (mm)    | d2 (mm)    | Total Path (mm)
--------------------------------------------------------------------------------
Lens 0.5Dia (35mm) | 35.0     | 135.95     | 36.94      | 172.89         
Lens 0.5Dia (40mm) | 40.0     | 212.31     | 43.31      | 255.62         
Lens 0.5Dia (50mm) | 50.0     | 326.57     | 55.32      | 381.89         
Lens 0.5Dia (60mm) | 60.0     | 425.64     | 67.03      | 492.67         
Lens 0.5Dia (75mm) | 75.0     | 563.87     | 84.40      | 648.27         
Lens 0.5Dia (100mm) | 100.0    | 783.01     | 113.13     | 896.14         
Lens 0.5Dia (125mm) | 125.0    | 996.21     | 141.75     | 1137.96        
Lens 0.5Dia (150mm) | 150.0    | 1206.65    | 170.32     | 1376.97        
Lens 0.5Dia (175mm) | 175.0    | 1415.58    | 198.86     | 1614.43        
Lens 0.5Dia (200mm) 